# 13 · Isolation Forest: does it catch attacks the detector misses?

Isolation Forest is an **unsupervised** model: it learns what ordinary requests look like and flags the ones that are easy to separate from the rest ("outliers"). It needs no attack labels, so in theory it could catch attack kinds nobody trained on.

Question: does it add anything to our XGBoost detector?

We try it two ways, on the same 27 site-independent `value_features` the detector uses:
1. **Fitted on public normal traffic** (CSIC + HttpParams normal rows, the same data the detector trains on), then applied to our lab logs.
2. **Fitted on each log itself** (no labels at all). This is how an outlier detector would run on a customer's log.

Data: development runs only (run3, run4 and camp7, both sites). The locked runs are not used. Every table shows the rules baseline and the detector next to Isolation Forest.

In [1]:
import sys
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from eval.match_labels import match
from eval.rules import rule_predict
from ml.detector import MODEL_PATH, to_frame
from ml.train_detector import HTTPPARAMS, load_httpparams
from ml.train_defense import DEV_RUNS

D = ROOT / 'data'
NAMES = ['run3 JS', 'run4 JS', 'run4 DVWA', 'camp7 JS', 'camp7 DVWA']
PAYLOAD = ['sqli', 'xss', 'traversal', 'cmdi']
BEHAVIOUR = ['scan', 'bruteforce', 'enum']

# Public normal traffic (the detector's training normals)
csic = pd.read_csv(D / 'processed' / 'csic_clean.csv', keep_default_na=False)
X = to_frame(csic['Method'], csic['path'], csic['body'])
y = csic['label']
Xh, yh = load_httpparams(HTTPPARAMS)
X, y = pd.concat([X, Xh], ignore_index=True), pd.concat([y, yh], ignore_index=True)
HAND = [c for c in X.columns if c != 'text']
public_normal = X.loc[y == 0, HAND].astype(float)
print(f'public normal requests: {len(public_normal)}, features: {len(HAND)}')

# Development runs, with the detector and rules decisions
detector = joblib.load(MODEL_PATH)
runs = {}
for name, (log, labels) in zip(NAMES, DEV_RUNS):
    df, report = match(D / 'raw' / log, D / 'labels' / labels)
    F = to_frame(df['method'], df['path'], df['body'])
    df['model'] = detector.predict(F)
    df['rules'] = rule_predict(df['method'], df['path'], df['body'])
    runs[name] = (df, F[HAND].astype(float))
    print(name, report)

public normal requests: 22514, features: 27


run3 JS {'log_lines': 4397, 'labels': 4397, 'matched': 4397, 'unmatched_log_lines': 0, 'unmatched_labels': 0}


run4 JS {'log_lines': 4296, 'labels': 4296, 'matched': 4296, 'unmatched_log_lines': 0, 'unmatched_labels': 0}


run4 DVWA {'log_lines': 3586, 'labels': 3567, 'matched': 3567, 'unmatched_log_lines': 19, 'unmatched_labels': 0}


camp7 JS {'log_lines': 6372, 'labels': 6372, 'matched': 6372, 'unmatched_log_lines': 0, 'unmatched_labels': 0}


camp7 DVWA {'log_lines': 6456, 'labels': 6387, 'matched': 6387, 'unmatched_log_lines': 69, 'unmatched_labels': 0}


In [2]:
def score(name, flagged):
    """One table row: false alarms, recall per kind, and what Isolation Forest adds to the detector."""
    df, _ = runs[name]
    normal = df['label'] == 'normal'
    payload = df['attack_type'].isin(PAYLOAD)
    missed = payload & (df['model'] == 0)
    row = {'run': name,
           'normal requests': int(normal.sum()),
           'IF false alarms': int((flagged & normal).sum()),
           'model false alarms': int(((df['model'] == 1) & normal).sum()),
           'rules payload recall': round(df.loc[payload, 'rules'].mean(), 3),
           'model payload recall': round(df.loc[payload, 'model'].mean(), 3),
           'IF payload recall': round(flagged[payload].mean(), 3),
           'missed by model': int(missed.sum()),
           'of those, caught by IF': int((flagged & missed).sum())}
    for t in BEHAVIOUR:
        kind = df['attack_type'] == t
        row[f'IF {t}'] = round(flagged[kind].mean(), 2) if kind.any() else None
    return row

## 1. Fitted on public normal traffic

The threshold is set on the public normals: a request is flagged when its outlier score is higher than 99%, 99.5% or 99.9% of them.

In [3]:
iso_public = IsolationForest(n_estimators=200, random_state=0, n_jobs=2).fit(public_normal)
train_scores = -iso_public.score_samples(public_normal)
tables = []
for q in [0.99, 0.995, 0.999]:
    thr = np.quantile(train_scores, q)
    rows = [score(n, pd.Series(-iso_public.score_samples(F) > thr, index=df.index))
            for n, (df, F) in runs.items()]
    t = pd.DataFrame(rows).set_index('run')
    t.insert(0, 'threshold', q)
    tables.append(t)
public_table = pd.concat(tables)
public_table

,threshold,normal requests,IF false alarms,model false alarms,rules payload recall,model payload recall,IF payload recall,missed by model,"of those, caught by IF",IF scan,IF bruteforce,IF enum
run,,,,,,,,,,,,
run3 JS,0.990,3328,2020,0,0.803,0.993,0.269,2,0,1.0,0.0,1.0
run4 JS,0.990,3269,1984,0,0.810,1.000,0.345,0,0,1.0,0.0,1.0
run4 DVWA,0.990,2588,1316,0,0.966,1.000,0.095,0,0,1.0,0.0,0.0
camp7 JS,0.990,3476,1998,0,0.807,0.994,0.292,6,0,1.0,0.0,1.0
camp7 DVWA,0.990,3515,1782,0,0.942,0.997,0.091,3,0,1.0,0.0,0.0
run3 JS,0.995,3328,1997,0,0.803,0.993,0.241,2,0,1.0,0.0,1.0
run4 JS,0.995,3269,1950,0,0.810,1.000,0.297,0,0,1.0,0.0,1.0
run4 DVWA,0.995,2588,1271,0,0.966,1.000,0.044,0,0,1.0,0.0,0.0
camp7 JS,0.995,3476,1976,0,0.807,0.994,0.248,6,0,1.0,0.0,1.0


**Reading:** at 99% and 99.5% Isolation Forest flags 49% to 61% of the **normal** lab requests. Our lab sites look different from the public data (different pages, fewer parameters), so to a model trained on public traffic they are all "outliers". At 99.9% the false alarms drop to 0, but it then catches under 1% of the attacks. This is the same lesson as notebook 04: a model that learns what *one* site's normal traffic looks like does not carry to another site.

## 2. Fitted on each log itself (no labels)

Each log gets its own Isolation Forest, fitted on all its requests without labels. `contamination` says which share of the log to flag (1%, 2% or 5%).

In [4]:
tables = []
for share in [0.01, 0.02, 0.05]:
    rows = []
    for n, (df, F) in runs.items():
        iso = IsolationForest(n_estimators=200, contamination=share, random_state=0, n_jobs=2).fit(F)
        rows.append(score(n, pd.Series(iso.predict(F) == -1, index=df.index)))
    t = pd.DataFrame(rows).set_index('run')
    t.insert(0, 'flag share', share)
    tables.append(t)
own_table = pd.concat(tables)
own_table

,flag share,normal requests,IF false alarms,model false alarms,rules payload recall,model payload recall,IF payload recall,missed by model,"of those, caught by IF",IF scan,IF bruteforce,IF enum
run,,,,,,,,,,,,
run3 JS,0.01,3328,0,0,0.803,0.993,0.134,2,0,0.00,0.0,0.0
run4 JS,0.01,3269,0,0,0.810,1.000,0.139,0,0,0.00,0.0,0.0
run4 DVWA,0.01,2588,0,0,0.966,1.000,0.122,0,0,0.00,0.0,0.0
camp7 JS,0.01,3476,0,0,0.807,0.994,0.066,6,0,0.00,0.0,0.0
camp7 DVWA,0.01,3515,0,0,0.942,0.997,0.068,3,0,0.00,0.0,0.0
run3 JS,0.02,3328,0,0,0.803,0.993,0.303,2,0,0.00,0.0,0.0
run4 JS,0.02,3269,0,0,0.810,1.000,0.277,0,0,0.00,0.0,0.0
run4 DVWA,0.02,2588,0,0,0.966,1.000,0.241,0,0,0.00,0.0,0.0
camp7 JS,0.02,3476,0,0,0.807,0.994,0.132,6,0,0.00,0.0,0.0


**Reading:** fitted on the log itself, Isolation Forest gives **0 false alarms** and finds part of the payload attacks (up to about 70% when it may flag 5% of the log), mostly XSS and SQL injection. But it **catches none of the attacks the detector missed**, and it finds no brute force or enumeration (these are normal-looking requests; the session layer finds them).

## 3. Why it cannot catch the detector's misses

Below: the features of every payload attack the detector missed in these runs (feature names only, no values).

In [5]:
rows = []
for n, (df, F) in runs.items():
    missed = df['attack_type'].isin(PAYLOAD) & (df['model'] == 0)
    for i in df.index[missed]:
        rows.append({'run': n, 'type': df.at[i, 'attack_type'], 'method': df.at[i, 'method'],
                     'request length': len(str(df.at[i, 'path'])) + len(str(df.at[i, 'body'])),
                     'non-zero features': ', '.join(c for c in HAND if F.at[i, c] != 0)})
pd.DataFrame(rows)

,run,type,method,request length,non-zero features
0,run3 JS,sqli,POST,61,"v_count, v_max_len, v_log_total_len, v_tautology"
1,run3 JS,cmdi,GET,49,"v_count, v_max_len, v_log_total_len"
2,camp7 JS,sqli,POST,55,"v_count, v_max_len, v_log_total_len, v_tautology"
3,camp7 JS,sqli,POST,61,"v_count, v_max_len, v_log_total_len, v_tautolo..."
4,camp7 JS,cmdi,POST,54,"v_count, v_max_len, v_log_total_len"
5,camp7 JS,sqli,POST,53,"v_count, v_max_len, v_log_total_len, v_tautology"
6,camp7 JS,cmdi,GET,51,"v_count, v_max_len, v_log_total_len"
7,camp7 JS,sqli,GET,40,"v_count, v_max_len, v_log_total_len, v_tautology"
8,camp7 DVWA,sqli,GET,56,"v_count, v_max_len, v_log_total_len, v_tautology"
9,camp7 DVWA,sqli,GET,56,"v_count, v_max_len, v_log_total_len, v_tautology"


The missed attacks are short values whose only signal is a weak tautology mark, or nothing at all. They look like ordinary values to every model that reads these features, so an outlier model cannot separate them either.

## Conclusion

- Isolation Forest **adds nothing** to the detector on our development runs: 0 extra attacks caught.
- Fitted on public data it flags a large share of normal traffic on new sites, so it is not site-independent.
- Fitted on the log itself it is safe (0 false alarms) but weaker than the detector.
- Decision (30 Oct): Isolation Forest does **not** change the detection. It is added as a separate **"unusual request, review it"** mark (`ml/outliers.py`): fitted on each uploaded log, top 1%. On these dev runs 0 normal requests were marked and every mark was already a detector alarm. Whether it finds attack kinds the detector misses is measured once, on run6 and the real-tool logs.
- For the report: an unsupervised model is the right tool for attack kinds nobody labelled; here the detector's site-independent features already capture what makes a request unusual, so there was little left for it to find.